In [1]:
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

with (project_root / "datasets" / "tinyshakespeare.txt").open() as f:
    text = f.read()

# vocab is the the list of "tokens" here chars but order needs to be consistent
# so can just use sorted
vocab = sorted(set(text))
vocab_size = len(vocab)
print(f"vocab size is {len(vocab)}")

vocab size is 65


In [2]:
char_to_id = {c: i for i, c in enumerate(vocab)}
id_to_char = {i: c for i, c in enumerate(vocab)}

# char -> token ids -> learned embedding (later )
# technically you could use chars but that'll be slow + can't use pytorch
# tensors would need an int and I just need some deterministic mapping
# can't use ord as then it expands to more than vocab size, indexes are 1 to 1
def encode(s: str) -> list[int]:
    return [char_to_id[c] for c in s]

# token ids -> chars
def decode(ids: list[int]) -> str:
    return "".join([id_to_char[i] for i in ids])

assert decode(encode("hello")) == "hello"

In [3]:
# now we make it a tensor
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using {device}")
data = torch.tensor(encode(text), device=device)
print(data.shape, data.dtype)

/home/zero/main/learning-ml/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


using cuda
torch.Size([1115394]) torch.int64


In [4]:
# train val split

train_ratio = 0.9
split_at = int(len(data) * train_ratio)
train, val = data[:split_at], data[split_at:]

assert(len(train) + len(val) == len(data))

In [5]:
# batching logic

# a batch is ( batch size, inputs ) , ( batch size, output )
# note that for a context len input say "First Cit"
# the input is 8 chars ( context size ) output is also 8 chars the next
# token
# First Ci => irst Cit
def get_batch(split, batch_size, context_size):
    last_valid_index = len(split) - context_size - 1
    positions = torch.randint(0, last_valid_index + 1, (batch_size,))
    chunks = [split[p:p+context_size+1] for p in positions]
    x_stacks = []
    y_stacks = []
    for c in chunks:
        x_stacks.append(c[:context_size])
        y_stacks.append(c[1:])
    return torch.stack(x_stacks), torch.stack(y_stacks)

CONTEXT_SIZE = 8
BATCH_SIZE = 32
x_batch, y_batch = get_batch(train, BATCH_SIZE, CONTEXT_SIZE)

print(f"x is {x_batch.shape}, y is {y_batch.shape}")
sample_no = 0
print(f"x[0] is {decode(x_batch[sample_no].tolist())}")
print(f"y[0] is {decode(y_batch[sample_no].tolist())}")

x is torch.Size([32, 8]), y is torch.Size([32, 8])
x[0] is ke civil
y[0] is e civil 


In [6]:
import torch.nn as nn

class BigramLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # token is a char here
        # internally embedding i just raw tensor + some pytorch wiring
        self.token_embedding = nn.Embedding(vocab_size, vocab_size)

    def forward(self, x_batch, y_batch = None):
        logits = self.token_embedding(x_batch)
        loss = None
        # only when you have a target -> training phase, that you can
        # compute loss
        if y_batch is not None:
            B, T, C = logits.shape
            # now I just care about the preds which are in C
            # merge B and position in batch = T
            y_preds = logits.view(-1, C)
            y_true = y_batch.view(-1)
            # cross entropy mathematically takes probs however this
            # specfic function in torch is meant to work with logits
            # and does the conversion to preds internally
            # applying twice aka passing it probabs here is wrong
            loss = nn.functional.cross_entropy(y_preds, y_true)
        return logits, loss

In [7]:
model = BigramLM(vocab_size).to(device)
x_batch, y_batch = get_batch(train, BATCH_SIZE, CONTEXT_SIZE)
logits, loss = model(x_batch, y_batch)
# loss is a scalar tensor, .item() makes it a float
print(f"Logit shape {logits.shape}, loss is {loss.item()}")

Logit shape torch.Size([32, 8, 65]), loss is 4.674842834472656


In [8]:
STEPS = 3000

optimizer = torch.optim.AdamW(model.parameters())
for step in range(STEPS):
    xb, yb = get_batch(train, BATCH_SIZE, CONTEXT_SIZE)
    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"step: {step}, loss: {loss.item()}")

step: 0, loss: 4.644626617431641
step: 100, loss: 4.597231864929199
step: 200, loss: 4.432323932647705
step: 300, loss: 4.341763496398926
step: 400, loss: 4.2053985595703125
step: 500, loss: 4.164712905883789
step: 600, loss: 4.077269554138184
step: 700, loss: 3.9700727462768555
step: 800, loss: 3.7955408096313477
step: 900, loss: 3.771306037902832
step: 1000, loss: 3.762789726257324
step: 1100, loss: 3.6670663356781006
step: 1200, loss: 3.5165014266967773
step: 1300, loss: 3.6155900955200195
step: 1400, loss: 3.3960962295532227
step: 1500, loss: 3.2907843589782715
step: 1600, loss: 3.3553099632263184
step: 1700, loss: 3.330763816833496
step: 1800, loss: 3.131476879119873
step: 1900, loss: 3.2521800994873047
step: 2000, loss: 3.1017003059387207
step: 2100, loss: 3.107088565826416
step: 2200, loss: 3.0422213077545166
step: 2300, loss: 3.015693187713623
step: 2400, loss: 2.9510021209716797
step: 2500, loss: 2.825470447540283
step: 2600, loss: 2.82304310798645
step: 2700, loss: 2.79827332

In [9]:
@torch.no_grad()
def estimate_loss(iters=200):
    out = {}
    model.eval() # redundant here
    for split_name, split_data in [("train", train), ("val", val)]:
        losses = torch.zeros(iters) # a iters sized 1d vec
        for iter in range(iters):
            xb, yb = get_batch(split_data, BATCH_SIZE, CONTEXT_SIZE)
            logits, loss = model(xb, yb)
            losses[iter] = loss.item()
        out[split_name] = losses.mean()
    model.train() # to avoid magic if I not run some other cell
    # this way the train -> eval -> train side effects remain local
    # to this only
    return out

loss_dict = estimate_loss()
print(f"loss {loss_dict}")

loss {'train': tensor(2.8013), 'val': tensor(2.8069)}


In [10]:
@torch.no_grad()
def generate(model, context, max_new_tokens):
    """context is (B, T) of token ids, returns (B, T + max_new_tokens).

    a plain function, not a method: works on any of the models below.
    """
    model.eval() # no dropout yet, but generating with dropout on is a
    # genuinely confusing bug once there is
    # a model can only see as many positions as its position table has rows.
    # ask the model instead of using a global -- new embeddings IS the thing
    # that would throw the IndexError, so it cannot go out of sync.
    # BigramLM has no position table and no limit at all.
    pos = getattr(model, "position_embedding", None)
    context_size = pos.num_embeddings if pos is not None else None

    for _ in range(max_new_tokens):
        # crop the INPUT only. context itself keeps growing since that's what
        # we decode at the end. slicing past the start is safe: a 5 token
        # context stays 5 tokens.
        window = context if context_size is None else context[:, -context_size:]

        logits, _ = model(window) # loss is None, no targets
        # it emits a prediction at every position, but only the newest one is
        # about a token we haven't seen yet
        logits_last = logits[:, -1, :] # B T C -> ( B, last token, C - logits )
        # B is 1 here but we could've done batched generations as well
        probs = nn.functional.softmax(logits_last, dim=-1)
        # sample, don't argmax. argmax is deterministic so from a given window
        # it always emits the same char and falls into a cycle
        next_id = torch.multinomial(probs, num_samples=1) # (B, 1)
        context = torch.cat((context, next_id), dim=1)

    model.train()
    return context

In [11]:
prompt = "hello"
context = torch.tensor(encode(prompt), device=device)
context = torch.unsqueeze(context,0) # to make it B, T -> input shape
# unsqueeze 0 -> add a 1 at 0th position

output = generate(model, context, 300)
print(decode(output[0].tolist()))

helloNGP&rv omien:-fYha?
SFse Is did ySvIMorcro gYoxOnoEAn Bs,Wqas,S:
NWakgpoI o SWha&weiscLvidKie pCBecist
LUSiq3, tthe y: t cozdm PI HqqqhZzrCKTpro thPriqm$s ShPElflds m?ayowicZ3rVMAry,OhewollJn worXOrmKwHELBOR:
NvNLA.vICE:WNDYjda nsvmeaie iP:
O: thNUaroraE: I h my,
d lofiklandigtele s, BrirVayteteaspc


In [12]:
# todo: compare against the freq count soln

In [13]:
import torch.nn as nn
import torch

N_EMBD = 32
# vocab size is the number of tokens

class LowRankBigramLM(nn.Module):
    def __init__(self, vocab_size, n_embd, context_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(context_size, n_embd)
        self.to_logits = nn.Linear(n_embd, vocab_size)

    def forward(self, x_batch, y_batch = None):
        B, T = x_batch.shape
        tok_embeds = self.token_embedding(x_batch) # B, T, N_EMBED
        # these only depend on positions so are constant in eval
        # but ofc get updated in training
        # tensor of [ 0,1,2,...,7 ]
        array_range = torch.arange(T, device=x_batch.device) # not "arrange" (T)
        pos_embeds = self.position_embedding(array_range) # T, N_EMBED
        # adds along last dims -> broadcasting
        x = tok_embeds + pos_embeds

        logits = self.to_logits(x)
        loss = None
        if y_batch is not None:
            C = logits.shape[-1] # vocab_size here, not n_embd
            y_preds = logits.view(-1, C)
            y_true = y_batch.view(-1)
            loss = nn.functional.cross_entropy(y_preds, y_true)
        return logits, loss

In [14]:
### ============== SAME CODE COPIED OVER

model = LowRankBigramLM(vocab_size, N_EMBD, CONTEXT_SIZE).to(device)
x_batch, y_batch = get_batch(train, BATCH_SIZE, CONTEXT_SIZE)
logits, loss = model(x_batch, y_batch)
print(f"Logit shape {logits.shape}, loss is {loss.item()}")

STEPS = 3000
# fresh optimizer: the old one still holds the bigram's tensors + its
# adam moment estimates, reusing it would update the wrong params
optimizer = torch.optim.AdamW(model.parameters())
for step in range(STEPS):
    xb, yb = get_batch(train, BATCH_SIZE, CONTEXT_SIZE)
    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"step: {step}, loss: {loss.item()}")

# bigram was 2.782 train / 2.783 val -- expect roughly the same, maybe a
# hair worse (rank 32 vs full rank 65x65)
print(estimate_loss())

Logit shape torch.Size([32, 8, 65]), loss is 4.610971450805664
step: 0, loss: 4.684476852416992
step: 100, loss: 3.5016560554504395
step: 200, loss: 3.1008682250976562
step: 300, loss: 2.9541163444519043
step: 400, loss: 2.8738327026367188
step: 500, loss: 2.6899032592773438
step: 600, loss: 2.576719284057617
step: 700, loss: 2.612362861633301
step: 800, loss: 2.813589334487915
step: 900, loss: 2.6071105003356934
step: 1000, loss: 2.6099815368652344
step: 1100, loss: 2.6208746433258057
step: 1200, loss: 2.517892599105835
step: 1300, loss: 2.6839752197265625
step: 1400, loss: 2.6897640228271484
step: 1500, loss: 2.492232084274292
step: 1600, loss: 2.538382053375244
step: 1700, loss: 2.5079402923583984
step: 1800, loss: 2.48999285697937
step: 1900, loss: 2.5275487899780273
step: 2000, loss: 2.7671191692352295
step: 2100, loss: 2.4798269271850586
step: 2200, loss: 2.4006752967834473
step: 2300, loss: 2.559842586517334
step: 2400, loss: 2.609241485595703
step: 2500, loss: 2.438048362731933

In [15]:
class Head(nn.Module):
    def __init__(self, n_embd, head_size, context_size):
        super().__init__()
        self.n_embd = n_embd
        self.head_size = head_size
        self.context_size = context_size
        # q k v are three linear maps applied to THE SAME x
        # same shape (n_embd -> head size), three SEPARATE learned matrices.
        # one shared matrix would make scores = x M M^T x = symmetric, so
        # "a attends to b" would force "b attends to a" equally. must not be.
        #
        # d_q MUST equal d_k = head_size, they get contracted against each
        # other. d_v is free (contracted against A, not against q) but
        # conventionally the same.
        #
        # bias=False: b_k is constant across a row so it cancels exactly in
        # the softmax, b_v is absorbed by any downstream bias. only b_q does
        # anything at all and the content path can synthesise it. dead params
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # tril = triangle lower, triu = upper.
        # the mask marks the BLOCKED cells (masked_fill writes where True), so
        # we want 1s above the diagonal. diagonal=1 is STRICTLY above => the
        # diagonal survives and a token can attend to itself. diagonal=0 would
        # blank row 0 entirely -> softmax over all -inf -> NaN
        # bool not float: masked_fill rejects float masks, and it's 4x smaller
        self.register_buffer("mask", torch.ones(context_size, context_size, dtype=torch.bool).triu(1))

    def forward(self, x_batch):
        B, T, C = x_batch.shape
        q, k, v = self.query(x_batch), self.key(x_batch), self.value(x_batch)

        # nn.Linear applies on the LAST dim only: (B, T, C=n_embd) -> (B, T, H)
        #
        # naming, same as the pytorch nn.MultiheadAttention docs:
        #   L = destination / target position -- comes from q, "who is asking"
        #   S = source position -- comes from k and v, "who is looked at"
        # here L == S == T because this is SELF attention: one tensor, every
        # position plays both roles. they genuinely differ in cross attention,
        # and at generation time with a kv-cache where L=1, S=all-so-far.
        #
        # q (B, L, H) @ k^T (B, H, S) -> (B, L, S),
        # transpose(a,b) what indexes in the shape to swap
        scores = q @ k.transpose(-2,-1) * (self.head_size ** -0.5)
        # [:T,:T] because T < context_size on a short prompt
        scores = scores.masked_fill(self.mask[:T, :T], -torch.inf)
        # A = attention matrix, (B, L, S)
        A = nn.functional.softmax(scores, dim=-1)
        # A (B, L, S) @ V (B, S, H) -> (B, L, H). S is contracted away.
        # out[i] = sum_j A[i,j] * v[j],
        out = A @ v
        return out

In [16]:
### ===== LowRankBigramLM COPIED OVER, only the two attention lines are new

class AttnLM(nn.Module):
    def __init__(self, vocab_size, n_embd, context_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(context_size, n_embd)
        # NEW head_size = n_embd so out drops straight into to_logits
        self.attention = Head(n_embd, n_embd, context_size)
        self.to_logits = nn.Linear(n_embd, vocab_size)

    def forward(self, x_batch, y_batch = None):
        B, T = x_batch.shape
        tok_embeds = self.token_embedding(x_batch) # B, T, N_EMBED
        # these only depend on positions so are constant in eval
        # but ofc get updated in training
        # tensor of [ 0,1,2,...,7 ]
        array_range = torch.arange(T, device=x_batch.device) # not "arrange" (T)
        pos_embeds = self.position_embedding(array_range) # T, N_EMBED
        # adds along last dims -> broadcasting
        x = tok_embeds + pos_embeds

        # NEW. up to here x[i] knows only token i and where it sits.
        # after this x[i] is a weighted blend of every position <= i
        x = self.attention(x) # B, T, n_embd

        logits = self.to_logits(x)
        loss = None
        if y_batch is not None:
            C = logits.shape[-1] # vocab_size here, not n_embd
            y_preds = logits.view(-1, C)
            y_true = y_batch.view(-1)
            loss = nn.functional.cross_entropy(y_preds, y_true)
        return logits, loss

In [17]:
### ===== SAME TRAINING CODE COPIED OVER, only the class name changed

model = AttnLM(vocab_size, N_EMBD, CONTEXT_SIZE).to(device)
x_batch, y_batch = get_batch(train, BATCH_SIZE, CONTEXT_SIZE)
logits, loss = model(x_batch, y_batch)
print(f"Logit shape {logits.shape}, loss is {loss.item()}")

STEPS = 3000
# fresh optimizer: the old one still holds the previous model's tensors + its
# adam moment estimates, reusing it would update the wrong params
optimizer = torch.optim.AdamW(model.parameters())
for step in range(STEPS):
    xb, yb = get_batch(train, BATCH_SIZE, CONTEXT_SIZE)
    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"step: {step}, loss: {loss.item()}")

# baselines to beat:
#   2.7820  bigram, undertrained at 3000 steps
#   2.5007  LowRankBigramLM control (train), 2.5078 val
#   2.4519  THE WALL -- exact counting bigram floor from freqs.
#   nothing that sees only
#   the current char can go below this. under it -> attention works
print(estimate_loss())

Logit shape torch.Size([32, 8, 65]), loss is 4.220764636993408
step: 0, loss: 4.168434143066406
step: 100, loss: 3.1629748344421387
step: 200, loss: 3.173953056335449
step: 300, loss: 2.9135704040527344
step: 400, loss: 2.970106363296509
step: 500, loss: 2.749711275100708
step: 600, loss: 2.754626512527466
step: 700, loss: 2.752582550048828
step: 800, loss: 2.848952531814575
step: 900, loss: 2.4919416904449463
step: 1000, loss: 2.610405921936035
step: 1100, loss: 2.5701801776885986
step: 1200, loss: 2.7308356761932373
step: 1300, loss: 2.416351795196533
step: 1400, loss: 2.3521463871002197
step: 1500, loss: 2.3871536254882812
step: 1600, loss: 2.4910573959350586
step: 1700, loss: 2.5112409591674805
step: 1800, loss: 2.363389015197754
step: 1900, loss: 2.543966770172119
step: 2000, loss: 2.3825387954711914
step: 2100, loss: 2.3574423789978027
step: 2200, loss: 2.520364999771118
step: 2300, loss: 2.325983762741089
step: 2400, loss: 2.5370888710021973
step: 2500, loss: 2.3921031951904297


In [18]:
# so now with just "one" attention we broke the 2.45 wall

In [19]:
# sanity checks, run generate and see what A the probabs are

# single head attention generate
prompt = "hello"
context = torch.unsqueeze(torch.tensor(encode(prompt), device=device), 0)
print(decode(generate(model, context, 4)[0].tolist()), end='\n\n')

# what did the single head learn
# A never leaves Head.forward, so rebuild it here from the trained
with torch.no_grad():
    # this is NOT on hello, we're changing samples
    xb, _ = get_batch(train, BATCH_SIZE, CONTEXT_SIZE)
    att = model.attention
    x = model.token_embedding(xb) + model.position_embedding(torch.arange(CONTEXT_SIZE, device=xb.device))
    q, k = att.query(x), att.key(x)
    scores = q @ k.transpose(-2, -1) * (att.head_size ** -0.5)
    scores = scores.masked_fill(att.mask[:CONTEXT_SIZE, :CONTEXT_SIZE], -torch.inf)
    A = nn.functional.softmax(scores, dim=-1) # B, L, S

# getting a probab matrix post the softmax
# 0 for the first batch, there is only 1 batch
chars = decode(xb[0].tolist())
print("sample 0:", repr(chars))
print("     " + " ".join(f"{c!r:>8}" for c in chars))
for i, row in enumerate(A[0].tolist()):
    print(f"{chars[i]!r:>5} " + " ".join(f"{v:8.3f}" for v in row))

hellocArk

sample 0: 'oved the'
          'o'      'v'      'e'      'd'      ' '      't'      'h'      'e'
  'o'    1.000    0.000    0.000    0.000    0.000    0.000    0.000    0.000
  'v'    0.004    0.996    0.000    0.000    0.000    0.000    0.000    0.000
  'e'    0.050    0.051    0.900    0.000    0.000    0.000    0.000    0.000
  'd'    0.028    0.072    0.349    0.552    0.000    0.000    0.000    0.000
  ' '    0.004    0.000    0.011    0.011    0.973    0.000    0.000    0.000
  't'    0.001    0.004    0.024    0.010    0.262    0.700    0.000    0.000
  'h'    0.000    0.001    0.001    0.002    0.039    0.053    0.905    0.000
  'e'    0.000    0.000    0.003    0.000    0.022    0.005    0.155    0.813


In [20]:
# big correction, I kep on saying probabs for A but that's no it
# it's a distribution of how much weight has attention given to each char
# in the contxt, which is why for the first char it's 1 as that's all there is
# to it

In [21]:
class FeedForward(nn.Module):
    def __init__(self, channel_dim, widening_factor = 4):
        # this init is what will do the setattr override
        # the parents setattrs will just assign this as module and won't
        # walk recursively so you need it at each level
        super().__init__()
        widened_dim = channel_dim * widening_factor
        # nn.Seq will register and so will me setting it as self.xyz =
        # it gets deduped by pytorch but no point in doing it twice
        self.ffn = nn.Sequential(
            nn.Linear(channel_dim,widened_dim),
            nn.ReLU(),
            nn.Linear(widened_dim, channel_dim)
            # rem no non-linearity at last
        )

    def forward(self, x):
        return self.ffn(x)

In [22]:
# same as earlier just FFN added

class AttnFFNLM(nn.Module):
    def __init__(self, vocab_size, n_embd, context_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(context_size, n_embd)
        # NEW. head_size = n_embd so out drops straight into to_logits
        self.attention = Head(n_embd, n_embd, context_size)
        self.ffn = FeedForward(n_embd, 4)
        self.to_logits = nn.Linear(n_embd, vocab_size)

    def forward(self, x_batch, y_batch = None):
        B, T = x_batch.shape
        tok_embeds = self.token_embedding(x_batch) # B, T, N_EMBED
        # these only depend on positions so are constant in eval
        # but ofc get updated in training
        # tensor of [ 0,1,2,...,7 ]
        array_range = torch.arange(T, device=x_batch.device) # not "arrange" (T)
        pos_embeds = self.position_embedding(array_range) # T, N_EMBED
        # adds along last dims -> broadcasting
        x = tok_embeds + pos_embeds

        # NEW. up to here x[i] knows only token i and where it sits.
        # after this x[i] is a weighted blend of every position <= i
        x = self.attention(x) # B, T, n_embd
        x = self.ffn(x)
        logits = self.to_logits(x)
        loss = None
        if y_batch is not None:
            C = logits.shape[-1] # vocab_size here, not n_embd
            y_preds = logits.view(-1, C)
            y_true = y_batch.view(-1)
            loss = nn.functional.cross_entropy(y_preds, y_true)
        return logits, loss

In [23]:
### ===== SAME TRAINING CODE COPIED OVER, only the class name changed

model = AttnFFNLM(vocab_size, N_EMBD, CONTEXT_SIZE).to(device)
x_batch, y_batch = get_batch(train, BATCH_SIZE, CONTEXT_SIZE)
logits, loss = model(x_batch, y_batch)
print(f"Logit shape {logits.shape}, loss is {loss.item()}")

STEPS = 3000
# fresh optimizer: the old one still holds the previous model's tensors + its
# adam moment estimates, reusing it would update the wrong params
optimizer = torch.optim.AdamW(model.parameters())
for step in range(STEPS):
    xb, yb = get_batch(train, BATCH_SIZE, CONTEXT_SIZE)
    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"step: {step}, loss: {loss.item()}")

# baselines to beat: 2.42 on val
print(estimate_loss())

Logit shape torch.Size([32, 8, 65]), loss is 4.219171524047852
step: 0, loss: 4.214820861816406
step: 100, loss: 3.031780958175659
step: 200, loss: 2.803337335586548
step: 300, loss: 2.6397252082824707
step: 400, loss: 2.4903879165649414
step: 500, loss: 2.7584362030029297
step: 600, loss: 2.676593065261841
step: 700, loss: 2.524697780609131
step: 800, loss: 2.567274570465088
step: 900, loss: 2.5117247104644775
step: 1000, loss: 2.410329818725586
step: 1100, loss: 2.507558822631836
step: 1200, loss: 2.462512254714966
step: 1300, loss: 2.532604694366455
step: 1400, loss: 2.5069761276245117
step: 1500, loss: 2.380182981491089
step: 1600, loss: 2.278226852416992
step: 1700, loss: 2.319660186767578
step: 1800, loss: 2.5577406883239746
step: 1900, loss: 2.3030495643615723
step: 2000, loss: 2.321230888366699
step: 2100, loss: 2.374824047088623
step: 2200, loss: 2.2786126136779785
step: 2300, loss: 2.2464852333068848
step: 2400, loss: 2.268561840057373
step: 2500, loss: 2.270632743835449
step

In [24]:
class MultiHead(nn.Module):
    # you don't need head_size but n_heads now, as size is derived
    def __init__(self, n_embd, n_heads, context_size):
        super().__init__()
        assert n_embd % n_heads == 0
        head_size = n_embd // n_heads

        # [A] = list comp, (A) is the gen, a "map" is not needed
        self.heads = nn.ModuleList(Head(n_embd, head_size, context_size) for _ in range(n_heads))
        self.lin_o = nn.Linear(n_embd, n_embd)

    def forward(self, x_batch):
        joined = torch.concat([head(x_batch) for head in self.heads], dim=-1)
        out = self.lin_o(joined)
        return out

In [25]:
class MultiHeadFFNLM(nn.Module):
    def __init__(self, vocab_size, n_embd, context_size, n_heads = 4):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(context_size, n_embd)
        self.mha = MultiHead(n_embd, n_heads, context_size)
        self.ffn = FeedForward(n_embd)
        self.to_logits = nn.Linear(n_embd, vocab_size)

    def forward(self, x_batch, y_batch = None):
        B, T = x_batch.shape
        tok_embeds = self.token_embedding(x_batch)
        array_range = torch.arange(T, device=x_batch.device) # not "arrange" (T)
        pos_embeds = self.position_embedding(array_range) # T, N_EMBED
        # adds along last dims -> broadcasting
        x = tok_embeds + pos_embeds

        x = self.mha(x) # B, T, n_embd
        x = self.ffn(x)
        logits = self.to_logits(x)
        loss = None
        if y_batch is not None:
            C = logits.shape[-1] # vocab_size here, not n_embd
            y_preds = logits.view(-1, C)
            y_true = y_batch.view(-1)
            loss = nn.functional.cross_entropy(y_preds, y_true)
        return logits, loss

In [26]:
model = MultiHeadFFNLM(vocab_size, N_EMBD, CONTEXT_SIZE).to(device)
x_batch, y_batch = get_batch(train, BATCH_SIZE, CONTEXT_SIZE)
logits, loss = model(x_batch, y_batch)
print(f"Logit shape {logits.shape}, loss is {loss.item()}")

STEPS = 3000
# fresh optimizer: the old one still holds the previous model's tensors + its
# adam moment estimates, reusing it would update the wrong params
optimizer = torch.optim.AdamW(model.parameters())
for step in range(STEPS):
    xb, yb = get_batch(train, BATCH_SIZE, CONTEXT_SIZE)
    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"step: {step}, loss: {loss.item()}")

# baselines to beat: 2.34 on val
print(estimate_loss())

Logit shape torch.Size([32, 8, 65]), loss is 4.1638994216918945
step: 0, loss: 4.141623497009277
step: 100, loss: 3.0689454078674316
step: 200, loss: 2.9142727851867676
step: 300, loss: 2.705435276031494
step: 400, loss: 2.7298879623413086
step: 500, loss: 2.5371193885803223
step: 600, loss: 2.4896528720855713
step: 700, loss: 2.4766507148742676
step: 800, loss: 2.3561511039733887
step: 900, loss: 2.3609867095947266
step: 1000, loss: 2.459698438644409
step: 1100, loss: 2.4957804679870605
step: 1200, loss: 2.4223103523254395
step: 1300, loss: 2.2297215461730957
step: 1400, loss: 2.540210247039795
step: 1500, loss: 2.4351489543914795
step: 1600, loss: 2.284620761871338
step: 1700, loss: 2.244661331176758
step: 1800, loss: 2.239069938659668
step: 1900, loss: 2.1895532608032227
step: 2000, loss: 2.3762385845184326
step: 2100, loss: 2.3190622329711914
step: 2200, loss: 2.3753700256347656
step: 2300, loss: 2.1072869300842285
step: 2400, loss: 2.076138734817505
step: 2500, loss: 2.24743938446

In [27]:
# generation test
prompt = "my"
context = torch.unsqueeze(torch.tensor(encode(prompt), device=device), 0)
print(decode(generate(model, context, 10)[0].tolist()), end='\n\n')

# so it's becoming SOMEwhat coherent now

my dous ade?



In [28]:
# it's called no res as no residual
class BlockNoRes(nn.Module):
    def __init__(self, n_embd, n_heads, context_size):
        super().__init__()
        self.mha = MultiHead(n_embd, n_heads, context_size)
        self.ffn = FeedForward(n_embd)

    def forward(self, x):
        x = self.mha(x)
        x = self.ffn(x)
        return x

In [29]:
class DeepMultiHeadFFNLM(nn.Module):
    def __init__(self, vocab_size, n_embd, context_size, n_heads, n_layers):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(context_size, n_embd)
        self.blocks = nn.ModuleList(BlockNoRes(n_embd, n_heads, context_size) for _ in range(n_layers))
        self.to_logits = nn.Linear(n_embd, vocab_size)

    def forward(self, x_batch, y_batch = None):
        B, T = x_batch.shape
        tok_embeds = self.token_embedding(x_batch)
        array_range = torch.arange(T, device=x_batch.device) # not "arrange" (T)
        pos_embeds = self.position_embedding(array_range) # T, N_EMBED
        # adds along last dims -> broadcasting
        x = tok_embeds + pos_embeds

        for b in self.blocks:
            x = b(x)
        logits = self.to_logits(x)
        loss = None
        if y_batch is not None:
            C = logits.shape[-1] # vocab_size here, not n_embd
            y_preds = logits.view(-1, C)
            y_true = y_batch.view(-1)
            loss = nn.functional.cross_entropy(y_preds, y_true)
        return logits, loss

In [30]:
# once again repeated training code
model = DeepMultiHeadFFNLM(vocab_size, N_EMBD, CONTEXT_SIZE, n_heads=4, n_layers=2).to(device)
x_batch, y_batch = get_batch(train, BATCH_SIZE, CONTEXT_SIZE)
logits, loss = model(x_batch, y_batch)
print(f"Logit shape {logits.shape}, loss is {loss.item()}")

STEPS = 3000
# fresh optimizer: the old one still holds the previous model's tensors + its
# adam moment estimates, reusing it would update the wrong params
optimizer = torch.optim.AdamW(model.parameters())
for step in range(STEPS):
    xb, yb = get_batch(train, BATCH_SIZE, CONTEXT_SIZE)
    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"step: {step}, loss: {loss.item()}")

# baselines to beat: 2.26 on val
print(estimate_loss())

Logit shape torch.Size([32, 8, 65]), loss is 4.199192047119141
step: 0, loss: 4.211884498596191
step: 100, loss: 3.3047823905944824
step: 200, loss: 3.112858533859253
step: 300, loss: 2.980276107788086
step: 400, loss: 2.7889328002929688
step: 500, loss: 2.8789477348327637
step: 600, loss: 2.641608238220215
step: 700, loss: 2.8010616302490234
step: 800, loss: 2.6654908657073975
step: 900, loss: 2.5537500381469727
step: 1000, loss: 2.4591076374053955
step: 1100, loss: 2.482200860977173
step: 1200, loss: 2.5179965496063232
step: 1300, loss: 2.5019690990448
step: 1400, loss: 2.4300527572631836
step: 1500, loss: 2.536026954650879
step: 1600, loss: 2.437875986099243
step: 1700, loss: 2.507288932800293
step: 1800, loss: 2.401289463043213
step: 1900, loss: 2.440835475921631
step: 2000, loss: 2.409364700317383
step: 2100, loss: 2.3938148021698
step: 2200, loss: 2.380617141723633
step: 2300, loss: 2.493194818496704
step: 2400, loss: 2.4671790599823
step: 2500, loss: 2.3211684226989746
step: 260

In [31]:
class Block(nn.Module):
    def __init__(self, n_embd, n_heads, context_size):
        super().__init__()
        self.mha = MultiHead(n_embd, n_heads, context_size)
        self.ffn = FeedForward(n_embd)

    def forward(self, x):
        x = x + self.mha(x)
        x = x + self.ffn(x)
        return x

In [32]:
# just a layer norm remains
class TransformerLMNoLayerNorm(nn.Module):
    def __init__(self, vocab_size, n_embd, context_size, n_heads, n_layers):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(context_size, n_embd)
        self.blocks = nn.ModuleList(Block(n_embd, n_heads, context_size) for _ in range(n_layers))
        self.to_logits = nn.Linear(n_embd, vocab_size)

    def forward(self, x_batch, y_batch = None):
        B, T = x_batch.shape
        tok_embeds = self.token_embedding(x_batch)
        array_range = torch.arange(T, device=x_batch.device) # not "arrange" (T)
        pos_embeds = self.position_embedding(array_range) # T, N_EMBED
        # adds along last dims -> broadcasting
        x = tok_embeds + pos_embeds

        for b in self.blocks:
            x = b(x)
        logits = self.to_logits(x)
        loss = None
        if y_batch is not None:
            C = logits.shape[-1] # vocab_size here, not n_embd
            y_preds = logits.view(-1, C)
            y_true = y_batch.view(-1)
            loss = nn.functional.cross_entropy(y_preds, y_true)
        return logits, loss

In [33]:
model = TransformerLMNoLayerNorm(vocab_size, N_EMBD, CONTEXT_SIZE, n_heads=4, n_layers=6).to(device) # back to 6 layers
x_batch, y_batch = get_batch(train, BATCH_SIZE, CONTEXT_SIZE)
logits, loss = model(x_batch, y_batch)
print(f"Logit shape {logits.shape}, loss is {loss.item()}")

STEPS = 3000
# fresh optimizer: the old one still holds the previous model's tensors + its
# adam moment estimates, reusing it would update the wrong params
optimizer = torch.optim.AdamW(model.parameters())
for step in range(STEPS):
    xb, yb = get_batch(train, BATCH_SIZE, CONTEXT_SIZE)
    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"step: {step}, loss: {loss.item()}")

# baselines to beat: 2.26 on val
print(estimate_loss())

Logit shape torch.Size([32, 8, 65]), loss is 4.907281398773193
step: 0, loss: 4.8754119873046875
step: 100, loss: 2.6937050819396973
step: 200, loss: 2.5524704456329346
step: 300, loss: 2.553086757659912
step: 400, loss: 2.285088062286377
step: 500, loss: 2.4214749336242676
step: 600, loss: 2.3109607696533203
step: 700, loss: 2.194812774658203
step: 800, loss: 2.261140823364258
step: 900, loss: 2.2878782749176025
step: 1000, loss: 2.3316900730133057
step: 1100, loss: 2.269385576248169
step: 1200, loss: 2.1094069480895996
step: 1300, loss: 2.175642251968384
step: 1400, loss: 2.347108840942383
step: 1500, loss: 2.1053013801574707
step: 1600, loss: 2.0870730876922607
step: 1700, loss: 2.080822467803955
step: 1800, loss: 1.8880987167358398
step: 1900, loss: 2.0669448375701904
step: 2000, loss: 1.9984021186828613
step: 2100, loss: 2.002711296081543
step: 2200, loss: 2.024284601211548
step: 2300, loss: 2.027106761932373
step: 2400, loss: 2.141441822052002
step: 2500, loss: 2.0652449131011963

In [34]:
# generation test, sub 2 on train now, ~2.1 on vals
# gibberish except ALMOST looks correct
prompt = "my"
context = torch.unsqueeze(torch.tensor(encode(prompt), device=device), 0)
print(decode(generate(model, context, 100)[0].tolist()), end='\n\n')

my word.

HEN VINTORKE:
Cime, come, unde i's favestn alll I kow flemphe will
setped knighwier to miday



In [35]:
class BlockNormed(nn.Module):
    def __init__(self, n_embd, n_heads, context_size):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mha = MultiHead(n_embd, n_heads, context_size)
        self.ffn = FeedForward(n_embd)

    def forward(self, x):
        x = x + self.mha(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class TransformerLM(nn.Module):
    def __init__(self, vocab_size, n_embd, context_size, n_heads, n_layers):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(context_size, n_embd)
        self.blocks = nn.ModuleList(BlockNormed(n_embd, n_heads, context_size) for _ in range(n_layers))
        self.ln_f = nn.LayerNorm(n_embd)
        self.to_logits = nn.Linear(n_embd, vocab_size)

    def forward(self, x_batch, y_batch = None):
        B, T = x_batch.shape
        tok_embeds = self.token_embedding(x_batch)
        array_range = torch.arange(T, device=x_batch.device) # not "arrange" (T)
        pos_embeds = self.position_embedding(array_range) # T, N_EMBED
        # adds along last dims -> broadcasting
        x = tok_embeds + pos_embeds

        for b in self.blocks:
            x = b(x)
        x = self.ln_f(x)
        logits = self.to_logits(x)
        loss = None
        if y_batch is not None:
            C = logits.shape[-1] # vocab_size here, not n_embd
            y_preds = logits.view(-1, C)
            y_true = y_batch.view(-1)
            loss = nn.functional.cross_entropy(y_preds, y_true)
        return logits, loss

In [36]:
model = TransformerLM(vocab_size, N_EMBD, CONTEXT_SIZE, n_heads=4, n_layers=6).to(device) # back to 6 layers
x_batch, y_batch = get_batch(train, BATCH_SIZE, CONTEXT_SIZE)
logits, loss = model(x_batch, y_batch)
print(f"Logit shape {logits.shape}, loss is {loss.item()}")

STEPS = 3000
# fresh optimizer: the old one still holds the previous model's tensors + its
# adam moment estimates, reusing it would update the wrong params
optimizer = torch.optim.AdamW(model.parameters())
for step in range(STEPS):
    xb, yb = get_batch(train, BATCH_SIZE, CONTEXT_SIZE)
    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"step: {step}, loss: {loss.item()}")

# baselines to beat: 2.09 now with lnorms
print(estimate_loss())

Logit shape torch.Size([32, 8, 65]), loss is 4.3960418701171875
step: 0, loss: 4.448185920715332
step: 100, loss: 2.951443672180176
step: 200, loss: 2.6064281463623047
step: 300, loss: 2.452749729156494
step: 400, loss: 2.4122519493103027
step: 500, loss: 2.3635318279266357
step: 600, loss: 2.3212037086486816
step: 700, loss: 2.2585058212280273
step: 800, loss: 2.114067316055298
step: 900, loss: 2.198251724243164
step: 1000, loss: 2.163987398147583
step: 1100, loss: 2.2743139266967773
step: 1200, loss: 2.1234374046325684
step: 1300, loss: 2.0919580459594727
step: 1400, loss: 2.0125715732574463
step: 1500, loss: 2.032055139541626
step: 1600, loss: 2.0799407958984375
step: 1700, loss: 2.1806893348693848
step: 1800, loss: 2.179226875305176
step: 1900, loss: 2.2220683097839355
step: 2000, loss: 2.0792417526245117
step: 2100, loss: 2.1424527168273926
step: 2200, loss: 2.1207785606384277
step: 2300, loss: 1.9923722743988037
step: 2400, loss: 1.973240613937378
step: 2500, loss: 2.070446968078